In [ ]:
import numpy
import random
import uuid
from algorithms.models import RPDHGParams, SolverConfig, GurobiParams
from algorithms.pdhg_restarted_cu import pdhg_restarted_cu
from algorithms.gurobi_lp import solve_lp_gurobi
from experiments.evaluation import evaluate_solver, warmup_rpdhg_gpu
from pathlib import Path
from dataclasses import replace
import transportation_problems
from transportation_problems.dotmark.loader_csv import load_dotmark_instance_csv
from transportation_problems.dotmark.build_ot_problem import build_ot_lp

def get_string_tuple_list(folder:str):
    string_Y = "transportation_problems/dotmark/csv_data/FOLDER/data32_XXXX.csv"
    string_X = string_Y.replace("FOLDER", folder)
    
    number_tuple_list = []
    for i in range(1,11):
        for j in range(i +1,11):
            number_tuple_list.append((i +1000, j+1000))
    
    string_tuple_list = []
    for tpl in number_tuple_list:
        string_A = string_X.replace("XXXX", str(tpl[0]))
        string_D = string_X.replace("XXXX", str(tpl[1]))
        string_tuple_list.append((string_A, string_D))
        
    return string_tuple_list

folder_names = ["MicroscopyImages","WhiteNoise","CauchyDensity","ClassicImages","GRFmoderate","GRFrough","GRFsmooth","LogitGRF","Shapes","LogGRF"]

# Pay for CUDA context init + CuPy kernel JIT compilation here, once, outside
# any timed run -- otherwise the first evaluate_solver(pdhg_restarted_cu, ...)
# call below would absorb that one-time cost into its measured runtime.
print("Warming up GPU (CUDA context + CuPy JIT)...")
warmup_rpdhg_gpu()

for folder_name in folder_names:

    for x in get_string_tuple_list(folder_name):
        path_A = Path(x[0])
        path_B = Path(x[1])

        print("Loading images...")
        Aimg, Bimg, a, b = load_dotmark_instance_csv(path_A, path_B)
        H, W = Aimg.shape

        print("Building OT LP...")

        from_number = x[0].split("/").pop()
        to_number = x[1].split("/").pop()

        good_name = folder_name + str(from_number) + "_to_" + str(to_number)

        dotmark_problem = build_ot_lp(a, b, H, W, name = good_name)

        print("LP size:", dotmark_problem.A.shape)


        tolerance = 1e-8
        my_params1 = RPDHGParams(
                        tau = None,
                        sigma = None,
                        theta = 1.03,
                        alpha= 1.0,
                        rebalancing_threshhold = 3.0,
                        step_shrinkage = 0.75,
                        restart_check = "adaptive" ,   #"adaptive" | "fixed" | "none"
                        min_epoch_length = 250,
                        max_iter = 100_000,
                        fixed_iter_restart= 3000,
                        tol_primal = tolerance,
                        tol_dual = tolerance,
                        tol_gap = tolerance,
                        tau_sigma_preconditioned= True,
                        rebalance_tau_sigma= True,
                        diagnostik_i = 25
        )


        directory_name = r"experiments\32 august 2"


        # Same target tolerance as rPDHG above (1e-8), and crossover=0 so
        # Gurobi stops at the barrier solution meeting that tolerance instead
        # of paying extra simplex-cleanup time for an exact vertex solution
        # rPDHG never produces either -- see conversation for rationale.
        gurobi_params = GurobiParams(
                        tol_primal = tolerance,
                        tol_dual = tolerance,
                        tol_gap = tolerance,
                        crossover = 0,
        )
        evaluate_solver(
                problem=dotmark_problem,
                solver_fn= lambda dotmark_problem: solve_lp_gurobi(dotmark_problem,gurobi_params),
                write_run=True,
                csv_export= False,
                exp_dir=directory_name,
                experiment_name=str(uuid.uuid4()),
                solver_config= SolverConfig(solver_name= "gurobi", params = gurobi_params)
            )

        # Full adaptive-restart rPDHG, same problem instance, same process,
        # run right after Gurobi above -- paired timing, same machine state.
        params = [my_params1]
        for p in params:

            evaluate_solver(
                problem=dotmark_problem,
                solver_fn= lambda dotmark_problem: pdhg_restarted_cu(dotmark_problem,p),
                write_run=True,
                csv_export= False,
                exp_dir=directory_name,
                experiment_name=str(uuid.uuid4()),
                solver_config= SolverConfig(solver_name= "rPDHG", params = p)
            )


C:\Users\Felix\PycharmProjects\rPDHGprivate\.venv\Lib\site-packages\cupy\_environment.py:275: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


Warming up GPU (CUDA context + CuPy JIT)...
Iter 20 | Primal: 6.7339e-01 | relPRes: 1.71e-02 | relDRes: 7.13e-03 | relGap: 5.68e-03|restartcheck: adaptive | tau: 1.366087990230169
Iter 40 | Primal: 6.0602e-01 | relPRes: 1.14e-02 | relDRes: 2.99e-03 | relGap: 8.48e-03|restartcheck: adaptive | tau: 0.9435230449560238
Loading images...
Building OT LP...
LP size: (2048, 1048576)
Set parameter Username
Set parameter LicenseID to value 2790741
Set parameter TimeLimit to value 600
Set parameter FeasibilityTol to value 1e-08
Set parameter OptimalityTol to value 1e-08
Set parameter Crossover to value 0
Academic license - for non-commercial use only - expires 2027-03-12
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: AMD Ryzen 5 3500X 6-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 6 logical processors, using up to 6 threads

Non-default parameters:
TimeLimit  600
FeasibilityTol  1e-08
OptimalityTol  1e-08
Crossover 